In [ ]:
# ============================================================
# Cell 1 — 사용자 설정
# ETF 과거 1분봉 수집·백테스트 (조회/연구 전용, 주문 기능 없음)
# ============================================================

RUN_MODE = "AUTO"                  # 권장: 전체 실행 한 번으로 수집범위 탐색→수집→백테스트
RUN_SELF_TESTS = True
KIWOOM_ENV = "REAL"                # "REAL" / "MOCK" (차트조회 지원 여부를 각각 실제 확인)
START_DATE = None                    # AUTO에서는 None: 키움이 제공하는 최초일까지 자동 탐색
END_DATE = None                      # AUTO에서는 None: 실행일 기준 최신 제공 데이터
BAR_MINUTES = "1"
OUTPUT_FORMAT = "csv"              # "csv" / "parquet"
PROJECT_ROOT = "."                 # 노트북을 실행한 현재 폴더에 data/outputs 생성

INDEX_TARGETS = {
    "001": {"name": "KOSPI", "market": "KOSPI"},
    "101": {"name": "KOSDAQ", "market": "KOSDAQ"},
    "201": {"name": "KOSPI200", "market": "KOSPI200"},
}
ETF_TARGETS = {
    "069500": {"name": "KODEX 200", "market": "KOSPI200", "role": "LONG_PRIMARY"},
    "102110": {"name": "TIGER 200", "market": "KOSPI200", "role": "LONG_COMPARE"},
    "105190": {"name": "ACE 200", "market": "KOSPI200", "role": "LONG_COMPARE"},
    "114800": {"name": "KODEX 인버스", "market": "KOSPI200", "role": "INVERSE_SHADOW"},
    "229200": {"name": "KODEX 코스닥150", "market": "KOSDAQ150", "role": "LONG_PRIMARY"},
    "232080": {"name": "TIGER 코스닥150", "market": "KOSDAQ150", "role": "LONG_COMPARE"},
    "354500": {"name": "ACE 코스닥150", "market": "KOSDAQ150", "role": "LONG_COMPARE"},
    "251340": {"name": "KODEX 코스닥150선물인버스", "market": "KOSDAQ150", "role": "INVERSE_SHADOW"},
}

PROBE_LOOKBACK_DAYS = [1, 30, 180, 365, 1095]
PROBE_PAGES_PER_DATE = 3
MAX_PAGES_PER_REQUEST = 100
AUTO_MAX_PAGES_PER_TARGET = 150      # 안전상한; 도달 시 PARTIAL로 보고하고 재실행 가능
API_MIN_INTERVAL_SEC = 0.35
HTTP_TIMEOUT_SEC = 20
MAX_RETRIES = 4
BACKOFF_BASE_SEC = 1.0
RAW_SAVE_ENABLED = True
RESUME_ENABLED = True

ORIGINAL_RULE_ENABLED = True
KOREAN_V20_RULE_ENABLED = True
ORIGINAL_ENTRY_TIME = "15:00"
ORIGINAL_TIME_EXIT = "15:20"
ETF_ENTRY_TIMES = ["14:30", "14:50", "15:00"]
ETF_TIME_EXIT = "15:20"
ETF_TP_PCTS = [0.30, 0.50, 0.75, 1.00]
ETF_SL_PCTS = [-0.20, -0.30, -0.50, -0.75]
AMBIGUOUS_BAR_POLICY = "SL_FIRST"
TIME_EXIT_MAX_STALENESS_MIN = 2      # 15:20 무체결 시 2분 이내 마지막 체결가 허용

ETF_COST_MODEL_VERSION = "etf_unconfirmed_v1"
ETF_COMMISSION_PCT_EACH_SIDE = None
ETF_TAX_PCT_SELL = None
ETF_SLIPPAGE_PCT_EACH_SIDE = None
ETF_FALLBACK_SLIPPAGE_TICKS_EACH_SIDE = 1
ETF_FALLBACK_TICK_WON = 5.0


In [ ]:
# ============================================================
# Cell 2 — 전체 본체
# ============================================================

import ast
import csv
import hashlib
import json
import os
import random
import tempfile
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
try:
    import requests
except ImportError:
    requests = None

KST = ZoneInfo("Asia/Seoul")
CHART_PATH = "/api/dostk/chart"
INDEX_API_ID = "ka20005"
ETF_API_ID = "ka10080"
ORDER_API_IDS = {"kt10000", "kt10001", "kt10002", "kt10003"}
ACCOUNT_API_IDS = {"ka10075", "kt00018"}
ACCESS_TOKEN = None
LAST_API_MONO = 0.0


# 연구 프로젝트 기준 경로와 출력 폴더를 준비합니다.
def resolve_paths():
    root = Path(PROJECT_ROOT).expanduser().resolve()
    paths = {
        "root": root,
        "raw": root / "data" / "raw",
        "processed": root / "data" / "processed",
        "outputs": root / "outputs",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


# 문자열 날짜를 date 객체로 엄격하게 변환합니다.
def parse_date(value):
    return datetime.strptime(str(value), "%Y-%m-%d").date()


# 키움 숫자 문자열의 부호·쉼표를 제거해 숫자로 변환합니다.
def parse_number(value, integer=False):
    if value is None or str(value).strip() == "":
        return None
    text = str(value).strip().replace(",", "")
    number = abs(float(text))
    return int(number) if integer else number


# 종목코드의 선행 0을 보존해 6자리 문자열로 정규화합니다.
def clean_code(value):
    text = str(value or "").strip()
    if text.isdigit() and len(text) == 3:
        return text
    return text.zfill(6) if text.isdigit() and len(text) <= 6 else text


# 여러 후보 키 중 응답에 실제 존재하는 첫 값을 반환합니다.
def pick(mapping, keys, default=None):
    for key in keys:
        if key in mapping and mapping[key] not in (None, ""):
            return mapping[key]
    return default


# .env를 출력하지 않고 현재 폴더와 프로젝트 상위에서 읽습니다.
def load_local_env(paths):
    candidates = [Path.cwd() / ".env", paths["root"] / ".env", paths["root"].parent.parent / ".env"]
    for env_path in candidates:
        if not env_path.exists():
            continue
        for raw in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            value = value.strip().strip('"').strip("'")
            os.environ.setdefault(key.strip(), value)
        return env_path
    return None


# 실행환경에 맞는 키움 REST 주소와 인증키를 가져옵니다.
def get_credentials(paths):
    load_local_env(paths)
    prefix = "KIWOOM_MOCK" if KIWOOM_ENV.upper() == "MOCK" else "KIWOOM"
    app_key = os.getenv(f"{prefix}_APP_KEY") or os.getenv("KIWOOM_APP_KEY", "")
    secret_key = os.getenv(f"{prefix}_SECRET_KEY") or os.getenv("KIWOOM_SECRET_KEY", "")
    default_url = "https://mockapi.kiwoom.com" if KIWOOM_ENV.upper() == "MOCK" else "https://api.kiwoom.com"
    base_url = os.getenv(f"{prefix}_BASE_URL") or os.getenv("KIWOOM_BASE_URL", default_url)
    if not app_key or not secret_key:
        raise RuntimeError("키움 APP_KEY/SECRET_KEY가 없습니다. .env를 확인하세요.")
    return base_url.rstrip("/"), app_key, secret_key


# JSON 파일을 임시파일과 replace 방식으로 원자 저장합니다.
def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, default=str)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(tmp_name, path)
    finally:
        if os.path.exists(tmp_name):
            os.unlink(tmp_name)


# CSV를 임시파일과 replace 방식으로 원자 저장합니다.
def atomic_write_dataframe(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = ".parquet" if path.suffix.lower() == ".parquet" else ".csv"
    fd, tmp_name = tempfile.mkstemp(prefix=f".{path.stem}.", suffix=suffix, dir=path.parent)
    os.close(fd)
    try:
        if suffix == ".parquet":
            frame.to_parquet(tmp_name, index=False)
        else:
            frame.to_csv(tmp_name, index=False, encoding="utf-8-sig")
        os.replace(tmp_name, path)
    finally:
        if os.path.exists(tmp_name):
            os.unlink(tmp_name)


# 기존 표와 신규 표를 합쳐 키 기준 중복 제거 후 원자 저장합니다.
def merge_save_table(path, new_frame, subset):
    path = Path(path)
    if path.exists():
        old = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path, dtype={"code": str})
        combined = pd.concat([old, new_frame], ignore_index=True)
    else:
        combined = new_frame.copy()
    combined["code"] = combined["code"].map(clean_code)
    combined = combined.drop_duplicates(subset=subset, keep="last").sort_values(subset).reset_index(drop=True)
    atomic_write_dataframe(path, combined)
    return combined


# 원응답 한 페이지를 덮어쓰지 않는 JSONL 파일로 보존합니다.
def append_raw_page(path, envelope):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    stable_envelope = {key: value for key, value in envelope.items() if key != "collected_at_kst"}
    fingerprint = hashlib.sha256(json.dumps(stable_envelope, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()
    row = {**envelope, "fingerprint": fingerprint}
    existing = set()
    if path.exists():
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                try:
                    existing.add(json.loads(line).get("fingerprint"))
                except Exception:
                    pass
    if fingerprint not in existing:
        with path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


# 전역 호출 간격을 지켜 조회 요청 시작을 제한합니다.
def wait_rate_limit():
    global LAST_API_MONO
    delay = API_MIN_INTERVAL_SEC - (time.monotonic() - LAST_API_MONO)
    if delay > 0:
        time.sleep(delay)
    LAST_API_MONO = time.monotonic()


# client_credentials 방식으로 접근토큰을 발급합니다.
def issue_token(session, base_url, app_key, secret_key):
    response = session.post(
        base_url + "/oauth2/token",
        json={"grant_type": "client_credentials", "appkey": app_key, "secretkey": secret_key},
        timeout=HTTP_TIMEOUT_SEC,
    )
    data = response.json()
    if response.status_code != 200 or int(data.get("return_code", 0)) != 0 or not data.get("token"):
        raise RuntimeError(f"토큰 발급 실패: HTTP {response.status_code} / {data.get('return_msg', data.get('message', ''))}")
    return data["token"]


# 주문·계좌 API를 차단한 상태로 키움 조회 POST를 재시도합니다.
def kiwoom_chart_post(session, base_url, app_key, secret_key, api_id, body, cont_yn="N", next_key=""):
    global ACCESS_TOKEN
    if api_id in ORDER_API_IDS | ACCOUNT_API_IDS or "/ordr" in CHART_PATH or "/acnt" in CHART_PATH:
        raise AssertionError("주문·계좌 API 호출은 이 연구 노트북에서 금지됩니다.")
    refreshed = False
    for attempt in range(MAX_RETRIES + 1):
        if not ACCESS_TOKEN:
            ACCESS_TOKEN = issue_token(session, base_url, app_key, secret_key)
        wait_rate_limit()
        headers = {
            "Content-Type": "application/json;charset=UTF-8",
            "authorization": f"Bearer {ACCESS_TOKEN}",
            "api-id": api_id,
            "cont-yn": cont_yn,
            "next-key": next_key,
        }
        try:
            response = session.post(base_url + CHART_PATH, headers=headers, json=body, timeout=HTTP_TIMEOUT_SEC)
            if response.status_code in (401, 403) and not refreshed:
                ACCESS_TOKEN = issue_token(session, base_url, app_key, secret_key)
                refreshed = True
                continue
            if response.status_code == 429 or response.status_code >= 500:
                if attempt >= MAX_RETRIES:
                    raise RuntimeError(f"HTTP {response.status_code}: 재시도 한도 초과")
                time.sleep(BACKOFF_BASE_SEC * (2 ** attempt) + random.uniform(0, 0.25))
                continue
            data = response.json()
            return_code = int(data.get("return_code", data.get("return_cd", 0)) or 0)
            if response.status_code != 200 or return_code != 0:
                raise RuntimeError(f"키움 오류 HTTP={response.status_code}, code={return_code}, msg={data.get('return_msg', data.get('message', ''))}")
            return data, response.headers
        except (requests.Timeout, requests.ConnectionError) as exc:
            if attempt >= MAX_RETRIES:
                raise RuntimeError(f"네트워크 재시도 한도 초과: {exc}") from exc
            time.sleep(BACKOFF_BASE_SEC * (2 ** attempt) + random.uniform(0, 0.25))
    raise RuntimeError("도달할 수 없는 요청 상태")


# API별 요청 본문을 공식 필드명으로 만듭니다.
def build_chart_body(instrument_type, code, base_date):
    ymd = base_date.strftime("%Y%m%d")
    if instrument_type == "INDEX":
        return INDEX_API_ID, {"inds_cd": str(code), "tic_scope": str(BAR_MINUTES), "base_dt": ymd}
    return ETF_API_ID, {"stk_cd": clean_code(code), "tic_scope": str(BAR_MINUTES), "upd_stkpc_tp": "1", "base_dt": ymd}


# 응답 객체에서 차트 행 배열을 유연하게 탐색합니다.
def extract_chart_rows(payload):
    preferred = [
        "stk_min_pole_chart_qry", "stk_min_pole_chart_qry_item", "stk_min_pole_chart",
        "inds_min_pole_chart_qry", "inds_min_pole_chart_qry_item", "inds_min_pole_chart",
        "output", "items", "data",
    ]
    for key in preferred:
        value = payload.get(key)
        if isinstance(value, list):
            return value, key
        if isinstance(value, dict):
            for nested in value.values():
                if isinstance(nested, list):
                    return nested, f"{key}.*"
    for key, value in payload.items():
        if isinstance(value, list) and (not value or isinstance(value[0], dict)):
            return value, key
    return [], ""


# 헤더와 본문 양쪽에서 연속조회 메타데이터를 읽습니다.
def parse_continuation(payload, headers):
    lower = {str(k).lower(): v for k, v in dict(headers).items()}
    cont = str(lower.get("cont-yn", payload.get("cont-yn", payload.get("cont_yn", "N")))).upper()
    next_key = str(lower.get("next-key", payload.get("next-key", payload.get("next_key", ""))) or "")
    return cont, next_key


# 키움 체결시간을 Asia/Seoul timezone-aware 시각으로 변환합니다.
def parse_kst_datetime(value, fallback_date):
    digits = "".join(ch for ch in str(value or "") if ch.isdigit())
    if len(digits) >= 14:
        dt = datetime.strptime(digits[:14], "%Y%m%d%H%M%S")
    elif len(digits) >= 12:
        dt = datetime.strptime(digits[:12], "%Y%m%d%H%M")
    elif len(digits) >= 6:
        dt = datetime.combine(fallback_date, datetime.strptime(digits[-6:], "%H%M%S").time())
    elif len(digits) >= 4:
        dt = datetime.combine(fallback_date, datetime.strptime(digits[-4:], "%H%M").time())
    else:
        raise ValueError(f"체결시간 해석 불가: {value}")
    return dt.replace(tzinfo=KST)


# 원시 차트 행을 공통 OHLCV 스키마로 정규화합니다.
def normalize_chart_rows(rows, instrument_type, code, meta, base_date, api_id, status="COMPLETE"):
    normalized = []
    time_keys = ["cntr_tm", "che_tm", "dt", "date", "datetime", "체결시간", "일자"]
    for raw in rows:
        try:
            dt = parse_kst_datetime(pick(raw, time_keys), base_date)
            close = parse_number(pick(raw, ["cur_prc", "close_pric", "close", "현재가", "종가"]))
            row = {
                "market": meta["market"], "instrument_type": instrument_type, "code": clean_code(code),
                "name": meta["name"], "datetime_kst": dt.isoformat(), "trade_date": dt.date().isoformat(),
                "open": parse_number(pick(raw, ["open_pric", "open", "시가"])),
                "high": parse_number(pick(raw, ["high_pric", "high", "고가"])),
                "low": parse_number(pick(raw, ["low_pric", "low", "저가"])),
                "close": close, "volume": parse_number(pick(raw, ["trde_qty", "volume", "거래량"]), integer=True),
                "cumulative_volume": parse_number(pick(raw, ["acc_trde_qty", "accumulated_trading_volume", "누적거래량"]), integer=True),
                "source_api": api_id, "adjusted_flag": "1" if instrument_type == "ETF" else "N/A",
                "collection_status": status,
            }
            normalized.append(row)
        except Exception:
            continue
    return normalized


# 단일 기준일의 연속조회 페이지를 수집하고 반복키를 차단합니다.
def fetch_chart_pages(session, credentials, paths, instrument_type, code, meta, base_date, page_limit, stop_date=None):
    base_url, app_key, secret_key = credentials
    api_id, body = build_chart_body(instrument_type, code, base_date)
    cont_yn, next_key, seen_keys = "N", "", set()
    all_rows, manifest_rows, final_status = [], [], "COMPLETE"
    for page in range(1, page_limit + 1):
        started = datetime.now(KST)
        try:
            payload, headers = kiwoom_chart_post(session, base_url, app_key, secret_key, api_id, body, cont_yn, next_key)
            rows, row_key = extract_chart_rows(payload)
            next_cont, new_key = parse_continuation(payload, headers)
            if RAW_SAVE_ENABLED:
                raw_name = f"kiwoom_{'sector' if instrument_type == 'INDEX' else 'etf'}_minute_{clean_code(code)}.jsonl"
                append_raw_page(paths["raw"] / raw_name, {
                    "collected_at_kst": started.isoformat(), "api_id": api_id, "code": clean_code(code),
                    "base_date": base_date.isoformat(), "page": page, "request_cont_yn": cont_yn,
                    "request_next_key": next_key, "response_cont_yn": next_cont, "response_next_key": new_key,
                    "return_code": payload.get("return_code", 0), "return_msg": payload.get("return_msg", ""),
                    "row_key": row_key, "response": payload,
                })
            all_rows.extend(rows)
            page_dates = []
            for raw_row in rows:
                try:
                    page_dates.append(parse_kst_datetime(pick(raw_row, ["cntr_tm", "che_tm", "dt", "date", "datetime", "체결시간", "일자"]), base_date).date())
                except Exception:
                    pass
            manifest_rows.append({
                "collected_at_kst": started.isoformat(), "api_id": api_id, "instrument_type": instrument_type,
                "code": clean_code(code), "base_date": base_date.isoformat(), "page": page,
                "row_count": len(rows), "cont_yn": next_cont, "next_key_present": bool(new_key), "status": "COMPLETE",
                "message": "", "first_raw_time": pick(rows[0], ["cntr_tm", "che_tm", "dt", "date"]) if rows else "",
                "last_raw_time": pick(rows[-1], ["cntr_tm", "che_tm", "dt", "date"]) if rows else "",
            })
            if stop_date is not None and page_dates and min(page_dates) <= stop_date:
                manifest_rows[-1]["message"] = "START_DATE_REACHED"
                break
            if next_cont != "Y":
                break
            if not new_key or new_key in seen_keys:
                final_status = "PARTIAL"
                manifest_rows[-1]["status"] = "PARTIAL"
                manifest_rows[-1]["message"] = "PAGINATION_LOOP_OR_MISSING_KEY"
                break
            seen_keys.add(new_key)
            cont_yn, next_key = "Y", new_key
        except Exception as exc:
            final_status = "API_ERROR"
            manifest_rows.append({
                "collected_at_kst": started.isoformat(), "api_id": api_id, "instrument_type": instrument_type,
                "code": clean_code(code), "base_date": base_date.isoformat(), "page": page, "row_count": 0,
                "cont_yn": cont_yn, "next_key_present": bool(next_key), "status": "API_ERROR", "message": str(exc)[:500],
                "first_raw_time": "", "last_raw_time": "",
            })
            break
    else:
        final_status = "PARTIAL"
        if manifest_rows:
            manifest_rows[-1]["status"] = "PARTIAL"
            manifest_rows[-1]["message"] = "PAGE_LIMIT_REACHED"
    if not all_rows and final_status == "COMPLETE":
        final_status = "NO_DATA"
        if manifest_rows:
            manifest_rows[-1]["status"] = "NO_DATA"
    normalized = normalize_chart_rows(all_rows, instrument_type, code, meta, base_date, api_id, final_status)
    if not normalized and final_status == "COMPLETE":
        final_status = "NO_DATA"
        if manifest_rows:
            manifest_rows[-1]["status"] = "NO_DATA"
            manifest_rows[-1]["message"] = manifest_rows[-1].get("message") or "NO_VALID_BARS"
    return normalized, manifest_rows, final_status


# 주말을 피한 탐색 기준일 목록을 최근·과거 순으로 만듭니다.
def make_probe_dates(today=None):
    anchor = today or datetime.now(KST).date()
    dates = []
    for days in PROBE_LOOKBACK_DAYS:
        candidate = anchor - timedelta(days=days)
        while candidate.weekday() >= 5:
            candidate -= timedelta(days=1)
        if candidate not in dates:
            dates.append(candidate)
    return dates


# 체크포인트를 읽어 완료된 수집 단위를 재개에서 건너뜁니다.
def load_checkpoint(paths):
    path = paths["raw"] / "collection_checkpoint.json"
    if RESUME_ENABLED and path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return {"completed": []}
    return {"completed": []}


# 수집 체크포인트를 원자적으로 갱신합니다.
def save_checkpoint(paths, checkpoint):
    atomic_write_json(paths["raw"] / "collection_checkpoint.json", checkpoint)


# 지정한 날짜·대상의 차트 데이터를 수집하고 manifest를 저장합니다.
def collect_targets(dates, page_limit, selected_targets=None, stop_date=None):
    if requests is None:
        raise RuntimeError("실제 수집에는 requests가 필요합니다. 현재 Python 환경에서 pip install requests를 실행하세요.")
    paths = resolve_paths()
    credentials = get_credentials(paths)
    checkpoint = load_checkpoint(paths)
    completed = set(checkpoint.get("completed", []))
    normalized_rows, manifest_rows = [], []
    with requests.Session() as session:
        targets = selected_targets or ([("INDEX", c, m) for c, m in INDEX_TARGETS.items()] + [("ETF", c, m) for c, m in ETF_TARGETS.items()])
        for base_date in dates:
            for instrument_type, code, meta in targets:
                unit = f"{instrument_type}|{clean_code(code)}|{base_date.isoformat()}|{page_limit}"
                if RESUME_ENABLED and unit in completed:
                    continue
                rows, pages, status = fetch_chart_pages(session, credentials, paths, instrument_type, code, meta, base_date, page_limit, stop_date=stop_date)
                if rows:
                    rows_frame = pd.DataFrame(rows)
                    if stop_date is not None:
                        rows_frame = rows_frame[pd.to_datetime(rows_frame["trade_date"]).dt.date >= stop_date]
                    rows = rows_frame.to_dict("records")
                normalized_rows.extend(rows)
                manifest_rows.extend(pages)
                if status in {"COMPLETE", "NO_DATA"}:
                    completed.add(unit)
                    checkpoint = {"updated_at_kst": datetime.now(KST).isoformat(), "completed": sorted(completed)}
                    save_checkpoint(paths, checkpoint)
                error_text = pages[-1].get("message", "") if pages else ""
                suffix = f" / {error_text}" if error_text else ""
                print(f"[{status}] {base_date} {instrument_type} {clean_code(code)} rows={len(rows)}{suffix}")
    if manifest_rows:
        manifest = pd.DataFrame(manifest_rows)
        manifest_path = paths["raw"] / "collection_manifest.csv"
        if manifest_path.exists():
            manifest = pd.concat([pd.read_csv(manifest_path, dtype={"code": str}), manifest], ignore_index=True)
        manifest = manifest.drop_duplicates(subset=["api_id", "code", "base_date", "page", "collected_at_kst"], keep="last")
        atomic_write_dataframe(manifest_path, manifest)
    if normalized_rows:
        frame = pd.DataFrame(normalized_rows)
        extension = ".parquet" if OUTPUT_FORMAT.lower() == "parquet" else ".csv"
        for instrument_type, filename in [("INDEX", "sector_minute_1m"), ("ETF", "etf_minute_1m")]:
            part = frame[frame["instrument_type"] == instrument_type]
            if not part.empty:
                merge_save_table(paths["processed"] / f"{filename}{extension}", part, ["source_api", "code", "datetime_kst"])
    return pd.DataFrame(normalized_rows), pd.DataFrame(manifest_rows)


# PROBE 결과에서 데이터 범위·페이지·상태를 요약합니다.
def build_probe_report(frame, manifest):
    if manifest.empty:
        return pd.DataFrame(columns=["instrument_type", "code", "status", "pages", "rows", "earliest", "latest"])
    ranges = pd.DataFrame(columns=["instrument_type", "code", "earliest", "latest"])
    if not frame.empty:
        ranges = frame.groupby(["instrument_type", "code"], as_index=False).agg(earliest=("datetime_kst", "min"), latest=("datetime_kst", "max"))
    summary = manifest.groupby(["instrument_type", "code"], as_index=False).agg(
        status=("status", lambda x: "API_ERROR" if "API_ERROR" in set(x) else ("PARTIAL" if "PARTIAL" in set(x) else ("COMPLETE" if x.str.contains("COMPLETE").any() else "NO_DATA"))),
        pages=("page", "count"), rows=("row_count", "sum"),
        message=("message", lambda x: " | ".join(sorted({str(v) for v in x if pd.notna(v) and str(v)}))[:1000]),
    )
    return summary.merge(ranges, on=["instrument_type", "code"], how="left")


# 처리 데이터의 중복·OHLC·거래량·장시간·분누락 품질을 검사합니다.
def quality_report(frame):
    columns = ["instrument_type", "code", "trade_date", "rows", "duplicates", "ohlc_errors", "negative_volume", "outside_session", "missing_minutes", "status"]
    if frame.empty:
        return pd.DataFrame(columns=columns)
    work = frame.copy()
    work["datetime_kst"] = pd.to_datetime(work["datetime_kst"], utc=True).dt.tz_convert("Asia/Seoul")
    reports = []
    for (instrument_type, code, trade_date), group in work.groupby(["instrument_type", "code", "trade_date"]):
        group = group.sort_values("datetime_kst")
        dup = int(group["datetime_kst"].duplicated().sum())
        valid = group.dropna(subset=["open", "high", "low", "close"])
        ohlc = int(((valid["high"] < valid[["open", "close", "low"]].max(axis=1)) | (valid["low"] > valid[["open", "close", "high"]].min(axis=1))).sum())
        neg_vol = int((pd.to_numeric(group["volume"], errors="coerce") < 0).sum())
        times = group["datetime_kst"].dt.strftime("%H:%M")
        outside = int(((times < "09:00") | (times > "15:30")).sum())
        unique_minutes = group["datetime_kst"].dt.floor("min").drop_duplicates().sort_values()
        gaps = unique_minutes.diff().dt.total_seconds().div(60)
        missing = int(gaps[gaps > 1].sub(1).sum()) if not gaps.empty else 0
        status = "FAIL" if dup or ohlc or neg_vol else ("WARN" if outside or missing else "PASS")
        reports.append([instrument_type, clean_code(code), trade_date, len(group), dup, ohlc, neg_vol, outside, missing, status])
    return pd.DataFrame(reports, columns=columns)


# 시각 이전 마지막 완성 분봉의 종가와 실제 시각을 반환합니다.
def price_at_or_before(day_frame, hhmm):
    cutoff = pd.Timestamp(f"{day_frame['trade_date'].iloc[0]} {hhmm}:00", tz="Asia/Seoul")
    eligible = day_frame[day_frame["datetime_kst"] <= cutoff]
    if eligible.empty:
        return None, None
    row = eligible.iloc[-1]
    return float(row["close"]), row["datetime_kst"]


# 신호 이후 첫 분봉 시가를 보수적인 진입가격으로 선택합니다.
def entry_after_signal(day_frame, hhmm):
    signal_time = pd.Timestamp(f"{day_frame['trade_date'].iloc[0]} {hhmm}:00", tz="Asia/Seoul")
    eligible = day_frame[day_frame["datetime_kst"] > signal_time]
    if eligible.empty:
        return None
    row = eligible.iloc[0]
    if row["datetime_kst"].floor("min") != signal_time + pd.Timedelta(minutes=1):
        return None
    return row


# 비용 설정을 검증하고 왕복 비용률과 확정상태를 반환합니다.
def resolve_cost_pct(entry_price, exit_price):
    known = all(v is not None for v in [ETF_COMMISSION_PCT_EACH_SIDE, ETF_TAX_PCT_SELL, ETF_SLIPPAGE_PCT_EACH_SIDE])
    if known:
        cost = 2 * ETF_COMMISSION_PCT_EACH_SIDE + ETF_TAX_PCT_SELL + 2 * ETF_SLIPPAGE_PCT_EACH_SIDE
        return float(cost), "CONFIRMED"
    tick_cost = 0.0
    if entry_price and exit_price and ETF_FALLBACK_SLIPPAGE_TICKS_EACH_SIDE:
        tick_cost = ETF_FALLBACK_SLIPPAGE_TICKS_EACH_SIDE * ETF_FALLBACK_TICK_WON * (1 / entry_price + 1 / exit_price) * 100
    return tick_cost, "UNCONFIRMED"


# 한 진입에서 TP·SL·시간청산 중 먼저 발생한 결과를 계산합니다.
def simulate_exit(day_frame, entry_row, tp_pct, sl_pct, time_exit):
    entry_price = float(entry_row["open"])
    tp_price = entry_price * (1 + tp_pct / 100)
    sl_price = entry_price * (1 + sl_pct / 100)
    cutoff = pd.Timestamp(f"{entry_row['trade_date']} {time_exit}:00", tz="Asia/Seoul")
    path = day_frame[(day_frame["datetime_kst"] >= entry_row["datetime_kst"]) & (day_frame["datetime_kst"] <= cutoff)]
    for _, bar in path.iterrows():
        hit_tp, hit_sl = float(bar["high"]) >= tp_price, float(bar["low"]) <= sl_price
        if hit_tp and hit_sl:
            exit_price = sl_price if AMBIGUOUS_BAR_POLICY == "SL_FIRST" else tp_price
            return exit_price, bar["datetime_kst"], "SL" if AMBIGUOUS_BAR_POLICY == "SL_FIRST" else "TP", True
        if hit_sl:
            return sl_price, bar["datetime_kst"], "SL", False
        if hit_tp:
            return tp_price, bar["datetime_kst"], "TP", False
    eligible = path[path["datetime_kst"] <= cutoff]
    if eligible.empty:
        return None, None, "DATA_UNAVAILABLE", False
    row = eligible.iloc[-1]
    staleness_min = (cutoff - row["datetime_kst"].floor("min")).total_seconds() / 60
    if staleness_min < 0 or staleness_min > TIME_EXIT_MAX_STALENESS_MIN:
        return None, None, "DATA_UNAVAILABLE", False
    return float(row["close"]), row["datetime_kst"], "TIME_EXIT", False


# v2.0 국내변형의 방향 결정을 미래정보 없이 계산합니다.
def decide_korean_direction(signal_day, entry_time):
    p0900, t0900 = price_at_or_before(signal_day, "09:00")
    p0930, t0930 = price_at_or_before(signal_day, "09:30")
    pentry, tentry = price_at_or_before(signal_day, entry_time)
    if None in (p0900, p0930, pentry) or t0900.strftime("%H:%M") != "09:00" or t0930.strftime("%H:%M") != "09:30" or tentry.strftime("%H:%M") != entry_time:
        return "", "DATA_UNAVAILABLE", None, None
    morning = (p0930 / p0900 - 1) * 100
    cumulative = (pentry / p0900 - 1) * 100
    if morning > 0 and cumulative > 0:
        return "LONG", "PASS", morning, cumulative
    if morning < 0 and cumulative < 0:
        return "INVERSE", "PASS", morning, cumulative
    return "", "NO_DIRECTION_CONSENSUS", morning, cumulative


# 방향별 주 실행 ETF 코드를 반환합니다.
def primary_etf_code(index_family, direction):
    mapping = {
        ("KOSPI200", "LONG"): "069500", ("KOSPI200", "INVERSE"): "114800",
        ("KOSDAQ150", "LONG"): "229200", ("KOSDAQ150", "INVERSE"): "251340",
    }
    return mapping.get((index_family, direction))


# 저장된 처리 데이터를 형식과 코드 보존 규칙에 맞춰 읽습니다.
def load_processed(paths):
    extension = ".parquet" if OUTPUT_FORMAT.lower() == "parquet" else ".csv"
    frames = []
    for stem in ["sector_minute_1m", "etf_minute_1m"]:
        path = paths["processed"] / f"{stem}{extension}"
        if path.exists():
            frame = pd.read_parquet(path) if extension == ".parquet" else pd.read_csv(path, dtype={"code": str})
            frames.append(frame)
    if not frames:
        raise FileNotFoundError("처리 분봉 파일이 없습니다. 먼저 PROBE/COLLECT를 실행하세요.")
    frame = pd.concat(frames, ignore_index=True)
    frame["code"] = frame["code"].map(clean_code)
    frame["datetime_kst"] = pd.to_datetime(frame["datetime_kst"], utc=True).dt.tz_convert("Asia/Seoul")
    for col in ["open", "high", "low", "close", "volume"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    return frame.sort_values(["code", "datetime_kst"]).drop_duplicates(["code", "datetime_kst"], keep="last")


# 원형 규칙과 국내변형 규칙을 분리해 거래·의사결정을 생성합니다.
def run_backtest(frame):
    etf = frame[frame["instrument_type"] == "ETF"].copy()
    decisions, trades = [], []
    dates = sorted(etf["trade_date"].dropna().unique())
    previous_close = {}
    for trade_date in dates:
        day_all = etf[etf["trade_date"] == trade_date]
        for family in ["KOSPI200", "KOSDAQ150"]:
            long_code = primary_etf_code(family, "LONG")
            signal_day = day_all[day_all["code"] == long_code].sort_values("datetime_kst")
            if signal_day.empty:
                continue
            policies = []
            if KOREAN_V20_RULE_ENABLED:
                for entry_time in ETF_ENTRY_TIMES:
                    direction, reason, morning, cumulative = decide_korean_direction(signal_day, entry_time)
                    policies.append(("KOREAN_V20", entry_time, direction, reason, morning, cumulative))
            if ORIGINAL_RULE_ENABLED:
                p0930, t0930 = price_at_or_before(signal_day, "09:30")
                prev = previous_close.get(long_code)
                if prev and p0930 and t0930.strftime("%H:%M") == "09:30":
                    first30 = (p0930 / prev - 1) * 100
                    direction = "LONG" if first30 > 0 else ("INVERSE" if first30 < 0 else "")
                    reason = "PASS" if direction else "NO_DIRECTION"
                else:
                    first30, direction, reason = None, "", "DATA_UNAVAILABLE"
                policies.append(("ORIGINAL", ORIGINAL_ENTRY_TIME, direction, reason, first30, None))
            for rule, entry_time, direction, reason, morning, cumulative in policies:
                decision_id = f"{trade_date}|{family}|{rule}|{entry_time}"
                decisions.append({
                    "decision_id": decision_id, "trade_date": trade_date, "index_family": family, "rule_family": rule,
                    "entry_time_variant": entry_time, "decision": "ENTER" if direction else "SKIP", "reason": reason,
                    "direction": direction, "morning_return_pct": morning, "cumulative_return_pct": cumulative,
                })
                code = primary_etf_code(family, direction)
                execution_day = day_all[day_all["code"] == code].sort_values("datetime_kst") if code else pd.DataFrame()
                entry_row = entry_after_signal(execution_day, entry_time) if not execution_day.empty else None
                if entry_row is None:
                    continue
                entry_id = decision_id + "|" + code
                for tp in ETF_TP_PCTS:
                    for sl in ETF_SL_PCTS:
                        exit_price, exit_time, exit_reason, ambiguous = simulate_exit(execution_day, entry_row, tp, sl, ETF_TIME_EXIT if rule == "KOREAN_V20" else ORIGINAL_TIME_EXIT)
                        if exit_price is None:
                            continue
                        entry_price = float(entry_row["open"])
                        gross = (exit_price / entry_price - 1) * 100
                        cost_pct, cost_status = resolve_cost_pct(entry_price, exit_price)
                        trades.append({
                            "entry_id": entry_id, "decision_id": decision_id, "trade_date": trade_date, "index_family": family,
                            "rule_family": rule, "entry_time_variant": entry_time, "direction": direction, "code": code,
                            "entry_datetime_kst": entry_row["datetime_kst"].isoformat(), "entry_price": entry_price,
                            "tp_pct": tp, "sl_pct": sl, "grid_family": "ETF_16", "exit_datetime_kst": exit_time.isoformat(),
                            "exit_price": exit_price, "exit_reason": exit_reason, "ambiguous_bar": ambiguous,
                            "gross_return_pct": gross, "estimated_cost_pct": cost_pct, "net_return_pct": gross - cost_pct,
                            "cost_status": cost_status, "cost_model_version": ETF_COST_MODEL_VERSION,
                        })
            close_price, close_time = price_at_or_before(signal_day, "15:30")
            if close_price is not None and close_time.strftime("%H:%M") == "15:30":
                previous_close[long_code] = close_price
    return pd.DataFrame(decisions), pd.DataFrame(trades)


# 16-grid 결과를 고유 진입수와 분리해 요약합니다.
def summarize_backtest(trades):
    if trades.empty:
        return pd.DataFrame()
    group_cols = ["rule_family", "index_family", "entry_time_variant", "direction", "tp_pct", "sl_pct", "cost_status"]
    rows = []
    for keys, group in trades.groupby(group_cols, dropna=False):
        gross = group["gross_return_pct"]
        net = group["net_return_pct"]
        gains = gross[gross > 0].sum()
        losses = -gross[gross < 0].sum()
        rows.append(dict(zip(group_cols, keys), samples=group["entry_id"].nunique(), grid_rows=len(group),
                         win_rate_pct=(gross > 0).mean() * 100, avg_gross_return_pct=gross.mean(),
                         avg_net_return_pct=net.mean(), gross_profit_factor=(gains / losses if losses > 0 else None)))
    return pd.DataFrame(rows)


# 거래별 결과를 거래일 단위로 요약합니다.
def summarize_daily(trades):
    if trades.empty:
        return pd.DataFrame()
    return trades.groupby(["trade_date", "rule_family", "index_family", "entry_time_variant", "tp_pct", "sl_pct"], as_index=False).agg(
        entries=("entry_id", "nunique"), gross_return_pct=("gross_return_pct", "mean"), net_return_pct=("net_return_pct", "mean")
    )


# fixture HTTP 응답 객체를 제공합니다.
class FixtureResponse:
    # fixture 응답 객체를 초기화합니다.
    def __init__(self, payload, headers=None, status_code=200):
        self.payload, self.headers, self.status_code = payload, headers or {}, status_code
        self.text = json.dumps(payload, ensure_ascii=False)

    # fixture 응답 JSON을 반환합니다.
    def json(self):
        return self.payload


# 정적·mock·경계 회귀테스트를 외부 연결 없이 실행합니다.
def run_self_tests():
    global ACCESS_TOKEN, LAST_API_MONO
    assert clean_code(69500) == "069500" and parse_number("-12,345") == 12345
    assert parse_kst_datetime("20260911143000", date(2026, 9, 11)).tzinfo is not None
    rows, key = extract_chart_rows({"stk_min_pole_chart_qry": [{"cntr_tm": "20260911143000"}]})
    assert len(rows) == 1 and key
    assert parse_continuation({}, {"cont-yn": "Y", "next-key": "abc"}) == ("Y", "abc")

    times = pd.date_range("2026-09-11 09:00", "2026-09-11 15:20", freq="min", tz="Asia/Seoul")
    prices = [100 + i * 0.01 for i in range(len(times))]
    fixture = pd.DataFrame({"trade_date": "2026-09-11", "datetime_kst": times, "open": prices, "high": [p + .1 for p in prices], "low": [p - .1 for p in prices], "close": prices})
    direction, reason, _, _ = decide_korean_direction(fixture, "14:30")
    assert direction == "LONG" and reason == "PASS"
    entry = entry_after_signal(fixture, "14:30")
    assert entry["datetime_kst"].strftime("%H:%M") == "14:31"
    ambiguous = fixture.iloc[[1]].copy()
    ambiguous.loc[:, "open"] = 100.0; ambiguous.loc[:, "high"] = 102.0; ambiguous.loc[:, "low"] = 98.0; ambiguous.loc[:, "close"] = 100.0
    ep, _, why, flag = simulate_exit(ambiguous, ambiguous.iloc[0], 1.0, -1.0, "15:20")
    assert why == "SL" and flag and ep == 99.0

    class NoNetworkSession:
        calls = []
        # 토큰과 차트 요청을 fixture로 응답합니다.
        def post(self, url, **kwargs):
            self.calls.append((url, kwargs.get("headers", {}).get("api-id")))
            if url.endswith("/oauth2/token"):
                return FixtureResponse({"return_code": 0, "token": "fixture"})
            return FixtureResponse({"return_code": 0, "stk_min_pole_chart_qry": [], "cont-yn": "N"})
    original_access_token, original_last_api_mono = ACCESS_TOKEN, LAST_API_MONO
    ACCESS_TOKEN, LAST_API_MONO = None, 0.0
    mock = NoNetworkSession()
    data, _ = kiwoom_chart_post(mock, "https://fixture.invalid", "a", "b", ETF_API_ID, {"stk_cd": "069500"})
    assert data["return_code"] == 0
    assert all(api_id not in ORDER_API_IDS | ACCOUNT_API_IDS for _, api_id in mock.calls if api_id)
    try:
        kiwoom_chart_post(mock, "https://fixture.invalid", "a", "b", "kt10001", {})
        raise AssertionError("주문 API 차단 실패")
    except AssertionError as exc:
        assert "금지" in str(exc)
    ACCESS_TOKEN, LAST_API_MONO = original_access_token, original_last_api_mono
    assert ACCESS_TOKEN == original_access_token

    grid = {(tp, sl) for tp in ETF_TP_PCTS for sl in ETF_SL_PCTS}
    assert len(grid) == 16
    print("정적·mock 회귀테스트: PASS")
    return True


# 실행 설정과 핵심 안전 불변식을 검사합니다.
def validate_settings():
    assert RUN_MODE in {"AUTO", "PROBE", "COLLECT", "BACKTEST"}
    assert KIWOOM_ENV in {"REAL", "MOCK"}
    assert str(BAR_MINUTES) == "1"
    assert OUTPUT_FORMAT in {"csv", "parquet"}
    assert AMBIGUOUS_BAR_POLICY in {"SL_FIRST", "TP_FIRST"}
    assert len(ETF_TP_PCTS) * len(ETF_SL_PCTS) == 16
    assert not (ORDER_API_IDS | ACCOUNT_API_IDS) & {INDEX_API_ID, ETF_API_ID}


# 실행 결과와 설정을 비밀정보 없이 manifest로 기록합니다.
def save_run_manifest(paths, started, status, details=None):
    payload = {
        "started_at_kst": started.isoformat(), "finished_at_kst": datetime.now(KST).isoformat(),
        "run_mode": RUN_MODE, "kiwoom_env": KIWOOM_ENV, "status": status,
        "bar_minutes": BAR_MINUTES, "start_date": START_DATE, "end_date": END_DATE,
        "original_rule_enabled": ORIGINAL_RULE_ENABLED, "korean_v20_rule_enabled": KOREAN_V20_RULE_ENABLED,
        "cost_model_version": ETF_COST_MODEL_VERSION, "details": details or {},
    }
    atomic_write_json(paths["outputs"] / "run_manifest.json", payload)


# PROBE·COLLECT·BACKTEST 모드를 분기해 전체 작업을 실행합니다.
def main():
    started = datetime.now(KST)
    paths = resolve_paths()
    print(f"[저장경로] {paths['root']}")
    try:
        validate_settings()
        if RUN_SELF_TESTS:
            run_self_tests()
        if RUN_MODE == "PROBE":
            frame, manifest = collect_targets(make_probe_dates(), PROBE_PAGES_PER_DATE)
            report = build_probe_report(frame, manifest)
            atomic_write_dataframe(paths["outputs"] / "probe_report.csv", report)
            if not frame.empty:
                atomic_write_dataframe(paths["outputs"] / "data_quality_report.csv", quality_report(frame))
            save_run_manifest(paths, started, "COMPLETE", {"probe_rows": len(frame), "manifest_rows": len(manifest)})
            display(report)
        elif RUN_MODE in {"AUTO", "COLLECT"}:
            end_date = parse_date(END_DATE) if END_DATE else datetime.now(KST).date()
            start_date = parse_date(START_DATE) if START_DATE else None
            targets = [("INDEX", c, m) for c, m in INDEX_TARGETS.items()] + [("ETF", c, m) for c, m in ETF_TARGETS.items()]
            page_limit = AUTO_MAX_PAGES_PER_TARGET if RUN_MODE == "AUTO" else MAX_PAGES_PER_REQUEST
            for target in targets:
                print(f"[자동수집 시작] {target[0]} {target[1]} {target[2]['name']}")
                collect_targets([end_date], page_limit, selected_targets=[target], stop_date=start_date)
            frame = load_processed(paths)
            if start_date is not None:
                frame = frame[pd.to_datetime(frame["trade_date"]).dt.date >= start_date]
            frame = frame[pd.to_datetime(frame["trade_date"]).dt.date <= end_date]
            quality = quality_report(frame)
            decisions, trades = run_backtest(frame)
            grid_summary, daily = summarize_backtest(trades), summarize_daily(trades)
            atomic_write_dataframe(paths["outputs"] / "data_quality_report.csv", quality)
            atomic_write_dataframe(paths["outputs"] / "signal_decisions.csv", decisions)
            atomic_write_dataframe(paths["outputs"] / "backtest_trades.csv", trades)
            atomic_write_dataframe(paths["outputs"] / "backtest_grid_summary.csv", grid_summary)
            atomic_write_dataframe(paths["outputs"] / "daily_summary.csv", daily)
            date_min = str(frame["trade_date"].min()) if not frame.empty else None
            date_max = str(frame["trade_date"].max()) if not frame.empty else None
            save_run_manifest(paths, started, "COMPLETE", {"available_start_date": date_min, "available_end_date": date_max, "processed_rows": len(frame), "unique_entries": int(trades["entry_id"].nunique()) if not trades.empty else 0, "grid_rows": len(trades)})
            print(f"[자동완료] 데이터범위 {date_min} ~ {date_max} / 유효진입 {trades['entry_id'].nunique() if not trades.empty else 0}건")
            display(grid_summary)
        else:
            frame = load_processed(paths)
            quality = quality_report(frame)
            decisions, trades = run_backtest(frame)
            grid_summary, daily = summarize_backtest(trades), summarize_daily(trades)
            atomic_write_dataframe(paths["outputs"] / "data_quality_report.csv", quality)
            atomic_write_dataframe(paths["outputs"] / "signal_decisions.csv", decisions)
            atomic_write_dataframe(paths["outputs"] / "backtest_trades.csv", trades)
            atomic_write_dataframe(paths["outputs"] / "backtest_grid_summary.csv", grid_summary)
            atomic_write_dataframe(paths["outputs"] / "daily_summary.csv", daily)
            save_run_manifest(paths, started, "COMPLETE", {"unique_entries": int(trades["entry_id"].nunique()) if not trades.empty else 0, "grid_rows": len(trades)})
            display(grid_summary)
    except Exception as exc:
        save_run_manifest(paths, started, "FAILED", {"error_type": type(exc).__name__, "error": str(exc)[:1000]})
        raise


if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# Cell 3 — QUICK REFERENCE (주석 전용)
# ============================================================
# 실행 순서
# 1) 권장: RUN_MODE="AUTO" 그대로 전체 셀 실행
# 2) 제공 가능 범위를 자동 탐색·수집하고 즉시 백테스트
# 3) outputs의 품질·거래·성과 요약파일 확인
# 4) 중단 시 같은 설정으로 다시 전체 실행하면 checkpoint 기반 재개
#
# 조회 API
# - ka20005 /api/dostk/chart: 지수 1분봉 (001 KOSPI, 101 KOSDAQ, 201 KOSPI200)
# - ka10080 /api/dostk/chart: ETF 1분봉 (8종, 수정주가구분 1)
# - 주문·계좌·잔고·미체결 API는 코드 차원에서 차단
#
# 주요 설정
# - RUN_MODE, START_DATE, END_DATE, INDEX_TARGETS, ETF_TARGETS
# - MAX_PAGES_PER_REQUEST, API_MIN_INTERVAL_SEC, MAX_RETRIES
# - ORIGINAL_RULE_ENABLED, KOREAN_V20_RULE_ENABLED
# - ETF_ENTRY_TIMES, ETF_TIME_EXIT, ETF_TP_PCTS, ETF_SL_PCTS
# - 비용은 수수료/세금/슬리피지를 독립 입력; 미확정이면 UNCONFIRMED
#
# 결과 파일
# - data/raw/collection_manifest.csv, collection_checkpoint.json, 원응답 JSONL
# - data/processed/sector_minute_1m.*, etf_minute_1m.*
# - outputs/probe_report.csv, data_quality_report.csv, signal_decisions.csv
# - outputs/backtest_trades.csv, backtest_grid_summary.csv, daily_summary.csv
# - outputs/run_manifest.json
#
# 백테스트 핵심
# - 원형: 전일 종가→09:30 방향, 15:00 신호 후 첫 분봉 진입
# - 국내변형: 09:00→09:30과 09:00→진입시각의 부호 합의
# - 14:30/14:50/15:00은 독립 counterfactual
# - 동일봉 TP·SL 동시 접촉은 기본 SL 우선 + ambiguous_bar=True
# - 16-grid는 1개 entry_id의 16개 정책 결과이며 거래 16건이 아님


In [ ]:
# ============================================================
# Cell 4 — PROJECT CONTINUITY NOTES (주석 전용)
# ============================================================
# 2026-09-14 (월)
# - Gao·Han·Li·Zhou(2018) 원형 가설과 v2.0 국내변형을 별도 rule_family로 구현.
# - 장중 v2.0과 프로세스·API·파일·상태를 공유하지 않는 독립 연구 노트북.
# - 기본값 PROBE. 실제 키움 보존기간·페이지 크기·KOSDAQ150 업종코드는 추정하지 않음.
# - 지수 001/101/201과 ETF 8종만 수집. 삼성전자·SK하이닉스는 초기 입력에서 제외.
# - ka20005/ka10080 조회만 허용하고 주문·계좌 API ID 및 경로를 명시적으로 차단.
# - 원응답 페이지 JSONL, manifest, atomic checkpoint, 중복제거 정규화 파일을 분리 저장.
# - 상태는 COMPLETE/NO_DATA/PARTIAL/API_ERROR로 구분하며 서버 미제공 데이터를 0으로 채우지 않음.
# - 신호시각 이후 첫 1분봉 시가를 진입가로 사용해 미래정보 누수를 방지.
# - TP/SL 동일봉 접촉은 보수적으로 SL 우선하며 모호봉 여부를 결과에 기록.
# - ETF 비용 미확정 시 주식 왕복비용 0.24%를 적용하지 않고 cost_status=UNCONFIRMED 유지.
# - 1차 정적·mock 검증 및 2차 비판적 리뷰 후 수정된 최종본에 전체 회귀검증 재실행.
# - 실제 조회 성공과 백테스트 분석 가능은 로컬 mock 통과와 별개로 PROBE 결과 확인 후 판정.
# - 최초 PROBE에서 mock fixture 토큰이 실제 요청에 잔류해 8005 오류가 난 결함을 수정.
# - 연구 결과 저장 기준을 노트북 실행 폴더로 변경하고, 지수 3자리 코드와 오류 원문 표시를 보완.
# - 사용자 요청에 따라 기본 실행을 AUTO로 변경: 전체 실행 한 번으로 가용범위 수집·품질검사·백테스트 완료.\n# - AUTO 수집은 종목별 최신 기준일에서 연속조회하며 날짜별 중복 전체조회를 제거.\n# - 15:20 무체결 경매구간은 2분 이내 마지막 체결가만 허용하고 실제 청산시각을 보존.\n